In [ ]:
def main(datasources, start_date, end_date):
    """
    构建 GP 当前最优因子。

    因子定义：
        -ts_std(intra_std(pct_change_m(high, periods=1)), window=5)

    平台会注入 datasources、start_date 和 end_date，并要求返回
    date、instrument、factor 三列的日频因子。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    output_start = pd.to_datetime(start_date).normalize()
    output_end = pd.to_datetime(end_date).normalize()

    # 5 个交易日滚动窗口需要测试区间之前的数据。
    query_start = output_start - pd.Timedelta(days=30)
    query_end = output_end + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    query_start_text = query_start.strftime("%Y-%m-%d %H:%M:%S")
    query_end_text = query_end.strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH base AS (
        SELECT
            date,
            date::DATE::DATETIME AS trading_day,
            instrument,
            high
        FROM {bar1m}
    ),
    minute_window AS (
        SELECT
            *,
            LAG(high, 1) OVER (
                PARTITION BY trading_day, instrument
                ORDER BY date
            ) AS previous_high
        FROM base
    ),
    daily AS (
        SELECT
            trading_day AS date,
            instrument,
            nanstd(high / NULLIF(previous_high, 0) - 1)
                AS intraday_high_return_std
        FROM minute_window
        GROUP BY trading_day, instrument
    )
    SELECT date, instrument, intraday_high_return_std
    FROM daily
    """

    daily = dai.query(
        sql,
        filters={"date": [query_start_text, query_end_text]},
        compression=True,
    ).df()
    daily["date"] = pd.to_datetime(daily["date"])
    daily["instrument"] = daily["instrument"].astype(str)
    daily["intraday_high_return_std"] = pd.to_numeric(
        daily["intraday_high_return_std"], errors="coerce"
    )

    # 先按扩展区间对齐中证 1000，再计算日频滚动窗口，与挖掘阶段一致。
    stock_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [query_start_text, query_end_text]},
    ).df()
    stock_pool["date"] = pd.to_datetime(stock_pool["date"])
    stock_pool["instrument"] = stock_pool["instrument"].astype(str)
    daily = pd.merge(
        daily, stock_pool, how="inner", on=["date", "instrument"]
    )
    daily = daily.sort_values(["instrument", "date"]).reset_index(drop=True)

    daily["factor"] = -daily.groupby(
        "instrument", sort=False
    )["intraday_high_return_std"].transform(
        lambda values: values.rolling(window=5, min_periods=5).std()
    )

    result = daily.loc[
        (daily["date"] >= output_start) & (daily["date"] <= output_end),
        ["date", "instrument", "factor"],
    ].copy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result["factor"] = result["factor"].replace(
        [np.inf, -np.inf], np.nan
    )
    result = result.dropna(subset=["factor"])
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)
